# 02 — Canonical Monthly Groundwater–InSAR Pairs

## Reproducible Groundwater–InSAR Workflow

This notebook reconstructs and verifies the monthly pairing and coverage-control stages used before lag analysis:

1. **Stage 04 — Groundwater–InSAR monthly pairing**
2. **Stage 05 — Piezometer coverage QC**
3. **Stage 06 — SBAS overlap increment QC**
4. **Stage 07 — Canonical monthly pairs**

The scientific calculations are implemented in version-controlled Python modules under `src/groundwater_insar/`.

The notebook documents the selection logic, quality-control rules, overlap behavior, and the final canonical dataset used by subsequent lag analyses.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from groundwater_insar.config import load_config
from groundwater_insar.stage04_monthly_pairs import reconstruct_stage04
from groundwater_insar.stage05_coverage_qc import reconstruct_stage05
from groundwater_insar.stage06_overlap_increment_qc import reconstruct_stage06
from groundwater_insar.stage07_canonical_pairs import reconstruct_stage07

cfg = load_config()

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("Repository  :", cfg["paths"]["repository_root"])
print("Workspace   :", cfg["paths"]["workspace_root"])
print("Regenerated :", cfg["paths"]["regenerated_root"])

## 1. Provenance

The current Git branch, commit, and working-tree state are recorded before reconstruction.  
The external notebook runner stores additional execution metadata in the notebook run manifest.

In [ ]:
repo = Path(cfg["paths"]["repository_root"])

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(repo), *args],
        text=True
    ).strip()

print("Git branch :", git("branch", "--show-current"))
print("Git commit :", git("rev-parse", "HEAD"))

status = git("status", "--porcelain")

print("Git dirty  :", bool(status))

if status:
    print("\nUncommitted files:")
    print(status)

## 2. Stage 04 — Monthly groundwater–InSAR pairing

Stage 03 defines the monthly InSAR time support.

Groundwater information from Stage 01 is joined using:

`Piezometer + JYear + JMonth + MonthIndex`

A monthly change pair is considered valid only when both groundwater and InSAR changes can be calculated. In operational terms, current and previous observations must exist for both variables.

In [ ]:
stage04, stage04_out, stage04_ref = reconstruct_stage04(cfg)

print("Generated :", stage04_out)
print("Reference :", stage04_ref)
print("Rows      :", len(stage04))
print("Wells     :", stage04["Piezometer"].nunique())

print(
    "Valid change pairs :",
    int(stage04["Valid_Change_Pair"].sum())
)

print(
    "Invalid pairs      :",
    int((~stage04["Valid_Change_Pair"]).sum())
)

display(stage04.head())

In [ ]:
pair_qc = pd.DataFrame({
    "Metric": [
        "Monthly rows",
        "Valid change pairs",
        "Invalid change pairs",
        "Missing GW declines",
        "Missing InSAR increments",
    ],
    "Value": [
        len(stage04),
        int(stage04["Valid_Change_Pair"].sum()),
        int((~stage04["Valid_Change_Pair"]).sum()),
        int(stage04["GW_Decline_m"].isna().sum()),
        int(stage04["InSAR_Increment_mm"].isna().sum()),
    ],
})

display(pair_qc)

## 3. Stage 05 — Piezometer coverage QC

Coverage is summarized separately for each official piezometer.

The reported quantities are:

- total monthly records;
- valid monthly InSAR observations;
- valid InSAR increments;
- valid groundwater declines;
- valid groundwater–InSAR change pairs;
- percentage of monthly records containing a valid change pair.

Piezometers without any usable InSAR change pairs are excluded from the subsequent canonical analysis.

In [ ]:
stage05, stage05_out, stage05_ref = reconstruct_stage05(cfg)

print("Generated :", stage05_out)
print("Reference :", stage05_ref)
print("Wells     :", len(stage05))

zero_pair_wells = stage05.loc[
    stage05["Valid_change_pairs"] == 0,
    "Piezometer"
].tolist()

eligible_wells = stage05.loc[
    stage05["Valid_change_pairs"] > 0,
    "Piezometer"
].tolist()

print("Zero-pair wells :", zero_pair_wells)
print("Eligible wells  :", len(eligible_wells))

display(stage05)

In [ ]:
coverage_plot = (
    stage05
    .sort_values(
        "Pair_coverage_pct",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(10, 8))

ax.barh(
    coverage_plot["Piezometer"],
    coverage_plot["Pair_coverage_pct"]
)

ax.set_xlabel("Valid monthly change-pair coverage (%)")
ax.set_ylabel("Piezometer")
ax.set_title("Groundwater–InSAR monthly pair coverage")

plt.tight_layout()
plt.show()

## 4. Stage 06 — Overlap increment QC

Some Jalali months are represented by two overlapping SBAS stacks.

The overlap QC retains only well–month combinations for which at least **two valid InSAR increments** are available.

For those cases, the two stack-derived increments are summarized using:

`N`, `Min`, `Max`, `Mean`, and `Range_mm`.

This check quantifies disagreement between independently derived increments at SBAS stack overlaps and avoids constructing increments across a stack reset.

In [ ]:
stage06, stage06_out, stage06_ref = reconstruct_stage06(cfg)

print("Generated :", stage06_out)
print("Reference :", stage06_ref)
print("Rows      :", len(stage06))
print("Wells     :", stage06["Piezometer"].nunique())

print(
    "Overlap MonthIndex values:",
    sorted(stage06["MonthIndex"].unique().tolist())
)

display(stage06.head(10))

In [ ]:
overlap_summary = stage06[
    ["Min", "Max", "Mean", "Range_mm"]
].describe()

display(overlap_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(
    stage06["Range_mm"],
    bins=15
)

ax.set_xlabel("Absolute range between overlapping stack increments (mm)")
ax.set_ylabel("Well-month groups")
ax.set_title("SBAS overlap increment disagreement")

plt.tight_layout()
plt.show()

## 5. Stage 07 — Canonical monthly pairs

Stage 07 produces one canonical record for each eligible:

`Piezometer × Jalali Month`

Only the official analysis period `1395–1403` is retained.

The canonical-selection rule is deterministic:

1. use a valid increment from the **same-year SBAS stack** when available;
2. otherwise use the nearest valid adjacent stack;
3. if no valid InSAR increment exists, retain the same-year stack when available, otherwise the nearest available stack.

For the current dataset, no canonical observation required the adjacent-valid fallback.

Derived fields are:

- `Valid_InSAR_Increment`
- `SameYearStack`
- `StackDistance`
- `SelectionRule`

In [ ]:
stage07, stage07_out, stage07_ref = reconstruct_stage07(cfg)

print("Generated :", stage07_out)
print("Reference :", stage07_ref)
print("Rows      :", len(stage07))
print("Wells     :", stage07["Piezometer"].nunique())

print(
    "Valid change pairs:",
    int(stage07["Valid_Change_Pair"].sum())
)

print("\nSelection rules:")
print(
    stage07["SelectionRule"]
    .value_counts()
    .to_string()
)

display(stage07.head())

In [ ]:
canonical_qc = pd.DataFrame({
    "Metric": [
        "Canonical rows",
        "Eligible wells",
        "Unique well-month keys",
        "Valid change pairs",
        "Invalid change pairs",
        "Same-year selected rows",
        "Adjacent-stack selected rows",
        "No-valid-increment rows",
    ],
    "Value": [
        len(stage07),
        stage07["Piezometer"].nunique(),
        stage07[
            ["Piezometer", "MonthIndex"]
        ].drop_duplicates().shape[0],
        int(stage07["Valid_Change_Pair"].sum()),
        int((~stage07["Valid_Change_Pair"]).sum()),
        int(
            (
                stage07["SelectionRule"]
                == "same_year_stack"
            ).sum()
        ),
        int(
            (
                stage07["SelectionRule"]
                == "adjacent_valid_stack"
            ).sum()
        ),
        int(
            (
                stage07["SelectionRule"]
                == "no_valid_insar_increment"
            ).sum()
        ),
    ],
})

display(canonical_qc)

In [ ]:
valid_by_year = (
    stage07[
        stage07["Valid_Change_Pair"]
    ]
    .groupby("JYear")
    .size()
    .rename("Valid_Change_Pairs")
    .reset_index()
)

display(valid_by_year)

print(
    "Total valid change pairs:",
    int(valid_by_year["Valid_Change_Pairs"].sum())
)

## 6. Frozen-baseline regression tests

Stages 04–07 are compared against the frozen terminal-analysis baseline.

Any scientific mismatch causes notebook execution to fail.

In [ ]:
regenerated_root = Path(
    cfg["paths"]["regenerated_root"]
)

regression_runs = [
    (
        4,
        regenerated_root / "stage04",
        "04_Stage04_MonthlyPairs_Regression.csv",
    ),
    (
        5,
        regenerated_root / "stage05",
        "05_Stage05_CoverageQC_Regression.csv",
    ),
    (
        6,
        regenerated_root / "stage06",
        "06_Stage06_OverlapIncrementQC_Regression.csv",
    ),
    (
        7,
        regenerated_root / "stage07",
        "07_Stage07_CanonicalPairs_Regression.csv",
    ),
]

for stage, candidate_dir, report_name in regression_runs:

    print("\n" + "=" * 90)
    print(f"REGRESSION — STAGE {stage:02d}")
    print("=" * 90)

    cmd = [
        sys.executable,
        "-m",
        "groundwater_insar.regression",
        "--candidate-dir",
        str(candidate_dir),
        "--stage",
        f"{stage:02d}",
        "--report-name",
        report_name,
    ]

    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Stage {stage:02d} regression failed."
        )

print("\nSTAGES 04–07 REGRESSION: PASS")

## 7. Scientific checkpoint

Passing this notebook establishes a reproducible monthly groundwater–InSAR dataset suitable for subsequent lag analysis.

At this checkpoint:

- 41 official piezometers were evaluated;
- four piezometers had no usable InSAR change pairs;
- 37 piezometers remain eligible;
- overlapping SBAS stacks were explicitly checked;
- one canonical observation was selected per eligible well and Jalali month;
- 3,504 canonical groundwater–InSAR change pairs are valid.

No lag interpretation is performed in this notebook.

In [ ]:
summary = pd.DataFrame({
    "Stage": [
        "04 — Monthly pairing",
        "05 — Coverage QC",
        "06 — Overlap increment QC",
        "07 — Canonical monthly pairs",
    ],
    "Rows": [
        len(stage04),
        len(stage05),
        len(stage06),
        len(stage07),
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
    ],
})

display(summary)

print()
print("02_MONTHLY_PAIRS NOTEBOOK: PASS")